### `Encryption`

In [ ]:
import os

def caesar(text, key):
    return ''.join(
        chr((ord(c)-65+key)%26+65) if c.isupper() else
        chr((ord(c)-97+key)%26+97) if c.islower() else c
        for c in text
    )

path = input("File path: ").strip()
if not os.path.isfile(path):
    print("File not found"); exit()

try:
    key = int(input("Key: "))
except:
    print("Invalid key"); exit()

text = open(path, encoding="utf-8").read()
out = path.rsplit('.',1)
out = f"{out[0]}_encrypted.{out[1]}" if len(out)>1 else path+"_encrypted"

open(out, "w", encoding="utf-8").write(caesar(text, key))
print("Saved to:", out)


### `Decryption`

In [ ]:
import os

def caesar(text, key):
    return ''.join(
        chr((ord(c)-65-key)%26+65) if c.isupper() else
        chr((ord(c)-97-key)%26+97) if c.islower() else c
        for c in text
    )

path = input("File path: ").strip()
if not os.path.isfile(path):
    print("File not found"); exit()

try:
    key = int(input("Key: "))
except:
    print("Invalid key"); exit()

text = open(path, encoding="utf-8").read()
out = path.rsplit('.',1)
out = f"{out[0]}_decrypted.{out[1]}" if len(out)>1 else path+"_decrypted"

open(out, "w", encoding="utf-8").write(caesar(text, key))
print("Saved to:", out)


### `Frequency Analysis Attack`

In [ ]:
import os
from collections import Counter

# English letter frequency
FREQ = {'a':8.167,'b':1.492,'c':2.782,'d':4.253,'e':12.702,'f':2.228,'g':2.015,'h':6.094,
'i':6.966,'j':0.153,'k':0.772,'l':4.025,'m':2.406,'n':6.749,'o':7.507,'p':1.929,'q':0.095,
'r':5.987,'s':6.327,'t':9.056,'u':2.758,'v':0.978,'w':2.360,'x':0.150,'y':1.974,'z':0.074}

# Caesar decrypt
def caesar(t,k):
    return ''.join(
        chr((ord(c)-65-k)%26+65) if c.isupper()
        else chr((ord(c)-97-k)%26+97) if c.islower()
        else c for c in t
    )

# Chi-squared score
def chi2(t):
    letters=[c.lower() for c in t if c.isalpha()]
    n=len(letters)
    if not n: return float('inf')
    count=Counter(letters)
    return sum(((count.get(c,0)-(FREQ[c]/100*n))**2)/(FREQ[c]/100*n) for c in FREQ)

# Get file
path=input("File path: ").strip()
if not os.path.isfile(path):
    print("File not found"); exit()

text=open(path,encoding="utf-8").read()

# Try all keys
results=sorted([(k,chi2(caesar(text,k)),caesar(text,k)) for k in range(26)], key=lambda x:x[1])

# Show top 5
print("\nTop 5 keys:")
for i,(k,s,d) in enumerate(results[:5],1):
    print(f"{i}. Key={k}, Score={s:.2f}, Preview={d[:50]}...")

# Best result
best_key, best_score, plaintext = results[0]

# Save with key in filename
base, ext = os.path.splitext(path)
out = f"{base}_attacked_key_{best_key}{ext}"
open(out,"w",encoding="utf-8").write(plaintext)

print(f"\nBest key: {best_key}")
print("Saved to:", out)


### `Report`

In [ ]:
import os
from collections import Counter

def load(path):
    if not os.path.isfile(path): print("File not found:", path); exit()
    return open(path, encoding="utf-8").read()

def compare(a, b):
    n = min(len(a), len(b))
    matches = sum(a[i]==b[i] for i in range(n))
    alpha = [(x,y) for x,y in zip(a[:n],b[:n]) if x.isalpha()]
    alpha_match = sum(x==y for x,y in alpha)
    non_alpha = [(x,y) for x,y in zip(a[:n],b[:n]) if not x.isalpha()]
    non_alpha_match = sum(x==y for x,y in non_alpha)

    word_a, word_b = a.split(), b.split()
    line_a, line_b = a.splitlines(), b.splitlines()

    report = []
    report.append("CAESAR DECRYPTION ACCURACY REPORT\n")
    report.append(f"Length match: {'YES' if len(a)==len(b) else 'NO'}")
    report.append(f"Overall accuracy: {matches/n*100:.2f}%")
    report.append(f"Alphabet accuracy: {alpha_match/len(alpha)*100:.2f}%" if alpha else "Alphabet accuracy: N/A")
    report.append(f"Non-alpha accuracy: {non_alpha_match/len(non_alpha)*100:.2f}%" if non_alpha else "Non-alpha accuracy: N/A")

    word_match = sum(x==y for x,y in zip(word_a,word_b))
    line_match = sum(x==y for x,y in zip(line_a,line_b))

    report.append(f"\nWord accuracy: {word_match/min(len(word_a),len(word_b))*100:.2f}%" if word_a and word_b else "")
    report.append(f"Line accuracy: {line_match/min(len(line_a),len(line_b))*100:.2f}%" if line_a and line_b else "")

    freq_a = Counter(c.lower() for c in a if c.isalpha())
    freq_b = Counter(c.lower() for c in b if c.isalpha())
    total_a, total_b = sum(freq_a.values()), sum(freq_b.values())

    report.append("\nLetter frequency diff:")
    for c in "abcdefghijklmnopqrstuvwxyz":
        pa = freq_a.get(c,0)/total_a*100 if total_a else 0
        pb = freq_b.get(c,0)/total_b*100 if total_b else 0
        report.append(f"{c}: {pb-pa:+.2f}%")

    report.append("\nVERDICT: " + (
        "PERFECT" if matches==n and len(a)==len(b) else
        "NEAR MATCH" if matches/n>=0.95 else
        "PARTIAL MATCH" if matches/n>=0.5 else
        "FAILED"
    ))

    return "\n".join(report)

orig = input("Original file: ").strip()
recv = input("Recovered file: ").strip()

a, b = load(orig), load(recv)
report = compare(a,b)

print("\n"+report)

out = orig.rsplit('.',1)
out = f"{out[0]}_comparison_report.txt"
open(out,"w",encoding="utf-8").write(report)

print("\nSaved to:", out)
